In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [01:10<00:00, 14.10s/it]


In [3]:
len(deals)

25

In [4]:
deals[10]

<Bird Buddy Smart Bird Feeder w/ Solar Roof for $180 + free shipping>

In [7]:
deals[10].describe()

"Title: Bird Buddy Smart Bird Feeder w/ Solar Roof for $180 + free shipping\nDetails: As one of Best Buy's daily deals, get this for $59 less than Amazon charges. My Best Buy members get free shipping. (It's free to join. Shipping is free for everyone over $35.)\xa0 Buy Now at Best Buy\nFeatures: \nURL: https://www.dealnews.com/Bird-Buddy-Smart-Bird-Feeder-w-Solar-Roof-for-180-free-shipping/21803655.html?iref=rss-f1912"

In [18]:
print(deals[1].describe())

Title: Unlocked Samsung Galaxy S25 Ultra Android Smart Phone: 512GB for 256GB price + up to $700 off w/ tra
Details: You can get the 512GB model for the price of the 256GB, or the 1TB model for the price of the 512GB. Without a trade-in, you can get $120 off the 512GB or $240 off the 1TB. With a trade-in, you also get up to an extra $700 off. Shop Now at Samsung
Features: 
URL: https://www.dealnews.com/Unlocked-Samsung-Galaxy-S25-Ultra-Android-Smart-Phone-512-GB-for-256-GB-price-up-to-700-off-w-tradef-free-shipping/21803884.html?iref=rss-c142


In [13]:
print(deals[10].describe())

Title: Bird Buddy Smart Bird Feeder w/ Solar Roof for $180 + free shipping
Details: As one of Best Buy's daily deals, get this for $59 less than Amazon charges. My Best Buy members get free shipping. (It's free to join. Shipping is free for everyone over $35.)  Buy Now at Best Buy
Features: 
URL: https://www.dealnews.com/Bird-Buddy-Smart-Bird-Feeder-w-Solar-Roof-for-180-free-shipping/21803655.html?iref=rss-f1912


In [17]:
print(deals[4].describe())

Title: Oukitel P2001 Plus Portable Power Station 2048Wh/2400W for $569 + free shipping
Details: Apply coupon code "OUPOWER100" for a savings of $100. Buy Now at oukitelpower.com
Features: 2,400W power output (4,800W peak surge output) supports a 1,800W max AC input and 500W max solar input LiFePo4 battery and BMS system
URL: https://www.dealnews.com/Oukitel-P2001-Plus-Portable-Power-Station-2048-Wh-2400-W-for-569-free-shipping/21803780.html?iref=rss-c142


### We are going to ask GPT-5-mini to summarize deals and identify their price

In [8]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.

Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.

**CRITICAL PRICING RULES:**
1. **"Off" / "Reduced by":** Be careful with products described as "$XXX off" or "reduced by $XXX" - this isn't the actual price. Only respond when the final checkout price is explicitly stated.
2. **"Up to":** EXCLUDE general sales events labeled as "Up to 70% off" or "Up to $1,800 off" unless a specific individual item with a specific numeric price is clearly listed.
3. **Trade-ins:** EXCLUDE deals that require a trade-in (e.g., "$700 off w/ trade-in"). We only want the direct purchase price without exchanging an old device.
4. **Upgrades:** If a deal says "512GB for 256GB price", extract it, but ensure the 'price' field is the actual dollar amount shown, not the value of the upgrade.
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.

**STRICT FILTERING CRITERIA:**
- **Ignore Trade-ins:** Do not include prices that depend on "w/ trade-in" or "with eligible plan".
- **Ignore "Up to" ranges:** Do not include generic sales like "Up to 70% off" or "Up to $1,800 off" if no specific item price is visible.
- **Clarify "Free Upgrade":** For deals like "512GB for 256GB price", record the price required to purchase the item.

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [9]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [10]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.

**STRICT FILTERING CRITERIA:**
- **Ignore Trade-ins:** Do not include prices that depend on "w/ trade-in" or "with eligible plan".
- **Ignore "Up to" ranges:** Do not include generic sales like "Up to 70% off" or "Up to $1,800 off" if no specific item price is visible.
- **Clarify "Free Upgrade":** For deals like "512GB for 256GB price", record the price required to purchase the item.

Deals:

Title: Newegg New Year, New Gear Sale: Up to 70% off + free shipping
Details: Save on gaming items, networking, smart home, software, automotive, and more. We've pictured the YLZKIX Gaming PC with A

In [11]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description="YLZKIX gaming desktop built around an AMD Ryzen 5 5600 (base 3.5GHz, boost up to 4.4GHz) paired with an AMD Radeon RX 6600/LE 8GB GPU. The system includes 16GB of RAM and a 1TB NVMe SSD, configured for modern 1080p gaming and general-purpose performance. It's a prebuilt desktop aimed at gamers who want a balance of CPU and midrange GPU power in a single tower.", price=790.0, url='https://www.dealnews.com/Newegg-New-Year-New-Gear-Sale-Up-to-70-off-free-shipping/21803888.html?iref=rss-c142'), Deal(product_description='Oukitel P2001 Plus is a portable power station featuring a 2,048Wh LiFePO4 battery and an advanced BMS for long cycle life and safety. It offers continuous 2,400W AC output with 4,800W peak surge capability, supports up to 1,800W AC input for recharging and 500W max solar input, and includes multiple AC and DC output ports for off-grid power, emergency backup, and outdoor use.', price=569.0, url='https://www.dealnews.com/Oukite

In [12]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


YLZKIX gaming desktop built around an AMD Ryzen 5 5600 (base 3.5GHz, boost up to 4.4GHz) paired with an AMD Radeon RX 6600/LE 8GB GPU. The system includes 16GB of RAM and a 1TB NVMe SSD, configured for modern 1080p gaming and general-purpose performance. It's a prebuilt desktop aimed at gamers who want a balance of CPU and midrange GPU power in a single tower.
790.0
https://www.dealnews.com/Newegg-New-Year-New-Gear-Sale-Up-to-70-off-free-shipping/21803888.html?iref=rss-c142

Oukitel P2001 Plus is a portable power station featuring a 2,048Wh LiFePO4 battery and an advanced BMS for long cycle life and safety. It offers continuous 2,400W AC output with 4,800W peak surge capability, supports up to 1,800W AC input for recharging and 500W max solar input, and includes multiple AC and DC output ports for off-grid power, emergency backup, and outdoor use.
569.0
https://www.dealnews.com/Oukitel-P2001-Plus-Portable-Power-Station-2048-Wh-2400-W-for-569-free-shipping/21803780.html?iref=rss-c142

N

### deal lưu ý: https://www.dealnews.com/Newegg-New-Year-New-Gear-Sale-Up-to-70-off-free-shipping/21803888.html


In [2]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [3]:
from agents.scanner_agent import ScannerAgent

In [4]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 25 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [5]:
result

DealSelection(deals=[Deal(product_description="YLZKIX gaming desktop powered by an AMD Ryzen 5 5600 (base 3.5GHz, boost up to 4.4GHz) paired with a Radeon RX 6600/LE 8GB GPU. The system includes 16GB of RAM and a 1TB NVMe SSD, built in a desktop chassis suitable for mid-range gaming and content creation. It's configured for Windows and typical gaming peripherals; the specs make it capable of 1080p gaming at medium-to-high settings.", price=790.0, url='https://www.dealnews.com/Newegg-New-Year-New-Gear-Sale-Up-to-70-off-free-shipping/21803888.html?iref=rss-c142'), Deal(product_description='Oukitel P2001 Plus is a portable power station featuring a 2,048 Wh LiFePO4 battery with integrated battery management system, a continuous 2,400W AC output (4,800W peak surge), and the ability to accept up to 1,800W AC input for charging plus 500W max solar input. It includes multiple AC outlets and DC/USB ports for powering appliances, tools, and charging devices during off-grid use or emergency back

In [6]:
load_dotenv(override=True)

True

In [7]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [8]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [9]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [12]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and Claude
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [13]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using Claude to craft the message
11:42:31 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
11:42:40 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
